# Task 1: Data Exploration and Enrichment

## Forecasting Financial Inclusion in Ethiopia

This notebook explores the unified financial inclusion dataset, validates its schema, identifies data-quality issues, and enriches it with additional observations, events, and impact relationships relevant to forecasting Access and Usage.

### Objectives

1. Load and inspect all provided datasets.
2. Understand the unified record structure.
3. Validate categorical fields using the reference codes.
4. Review observations, events, targets, and impact links.
5. Identify missing or inconsistent information.
6. Add relevant records using credible sources.
7. Export a cleaned and enriched analysis-ready dataset.

In [1]:
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_DATA_PATH = RAW_DATA_DIR / "ethiopia_fi_unified_data.xlsx"
REFERENCE_CODES_PATH = RAW_DATA_DIR / "reference_codes.xlsx"
GUIDE_PATH = RAW_DATA_DIR / "Additional Data Points Guide.xlsx"

print("Project root:", PROJECT_ROOT)
print("Unified data exists:", UNIFIED_DATA_PATH.exists())
print("Reference codes exist:", REFERENCE_CODES_PATH.exists())
print("Enrichment guide exists:", GUIDE_PATH.exists())

Project root: c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast
Unified data exists: True
Reference codes exist: True
Enrichment guide exists: True


In [3]:
unified_workbook = pd.ExcelFile(UNIFIED_DATA_PATH)
reference_workbook = pd.ExcelFile(REFERENCE_CODES_PATH)
guide_workbook = pd.ExcelFile(GUIDE_PATH)

print("Unified workbook sheets:")
print(unified_workbook.sheet_names)

print("\nReference-code workbook sheets:")
print(reference_workbook.sheet_names)

print("\nAdditional Data Points Guide sheets:")
print(guide_workbook.sheet_names)

Unified workbook sheets:
['ethiopia_fi_unified_data', 'Impact_sheet']

Reference-code workbook sheets:
['reference_codes']

Additional Data Points Guide sheets:
['A. Alternative Baselines', 'B. Direct Corrln', 'C. Indirect Corrln', 'D. Market Naunces']


In [4]:
main_df = pd.read_excel(
    UNIFIED_DATA_PATH,
    sheet_name="ethiopia_fi_unified_data"
)

impact_df = pd.read_excel(
    UNIFIED_DATA_PATH,
    sheet_name="Impact_sheet"
)

reference_codes_df = pd.read_excel(
    REFERENCE_CODES_PATH,
    sheet_name="reference_codes"
)

alternative_baselines_df = pd.read_excel(
    GUIDE_PATH,
    sheet_name="A. Alternative Baselines",
    header=8
)

direct_indicators_df = pd.read_excel(
    GUIDE_PATH,
    sheet_name="B. Direct Corrln",
    header=9
)

indirect_indicators_df = pd.read_excel(
    GUIDE_PATH,
    sheet_name="C. Indirect Corrln",
    header=9
)

market_nuances_df = pd.read_excel(
    GUIDE_PATH,
    sheet_name="D. Market Naunces",
    header=7
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [5]:
datasets = {
    "Main data": main_df,
    "Impact links": impact_df,
    "Reference codes": reference_codes_df,
    "Alternative baselines": alternative_baselines_df,
    "Direct indicators": direct_indicators_df,
    "Indirect indicators": indirect_indicators_df,
    "Market nuances": market_nuances_df,
}

for name, dataframe in datasets.items():
    print(f"{name:<25}: {dataframe.shape[0]} rows × {dataframe.shape[1]} columns")

Main data                : 43 rows × 34 columns
Impact links             : 14 rows × 35 columns
Reference codes          : 71 rows × 4 columns
Alternative baselines    : 10 rows × 8 columns
Direct indicators        : 19 rows × 6 columns
Indirect indicators      : 16 rows × 6 columns
Market nuances           : 10 rows × 7 columns


In [6]:
all_columns = sorted(set(main_df.columns).union(set(impact_df.columns)))

main_aligned = main_df.reindex(columns=all_columns)
impact_aligned = impact_df.reindex(columns=all_columns)

unified_df = pd.concat(
    [main_aligned, impact_aligned],
    ignore_index=True
)

print("Combined unified dataset shape:", unified_df.shape)
print("Total records:", len(unified_df))

Combined unified dataset shape: (57, 35)
Total records: 57


In [7]:
record_type_counts = (
    unified_df["record_type"]
    .value_counts(dropna=False)
    .rename_axis("record_type")
    .reset_index(name="record_count")
)

record_type_counts

,record_type,record_count
0,observation,30
1,impact_link,14
2,event,10
3,target,3


In [8]:
print("Duplicate record IDs:", unified_df["record_id"].duplicated().sum())
print("Missing record IDs:", unified_df["record_id"].isna().sum())
print("Missing record types:", unified_df["record_type"].isna().sum())

event_records = unified_df[unified_df["record_type"] == "event"]
impact_records = unified_df[unified_df["record_type"] == "impact_link"]

print("\nEvents with incorrectly populated pillar:")
print(event_records[event_records["pillar"].notna()][
    ["record_id", "indicator", "category", "pillar"]
])

print("\nImpact links with missing parent ID:")
print(impact_records[impact_records["parent_id"].isna()][
    ["record_id", "indicator", "parent_id"]
])

Duplicate record IDs: 0
Missing record IDs: 0
Missing record types: 0

Events with incorrectly populated pillar:
Empty DataFrame
Columns: [record_id, indicator, category, pillar]
Index: []

Impact links with missing parent ID:
Empty DataFrame
Columns: [record_id, indicator, parent_id]
Index: []


## 2. Dataset Structure and Record-Type Analysis

The unified schema stores observations, events, policy targets, and modeled event-impact relationships in a consistent structure. Record types must be interpreted differently:

- **Observations** contain measured values for specific indicators.
- **Events** describe policies, launches, infrastructure developments, and market milestones.
- **Targets** contain official policy goals.
- **Impact links** connect neutral events to indicators they may affect.

Events are intentionally not assigned directly to a pillar. Their effects are represented through impact-link records to avoid forcing a biased interpretation onto the raw event data.

In [9]:
for record_type in ["observation", "event", "target", "impact_link"]:
    print(f"\n{'=' * 80}")
    print(f"EXAMPLE: {record_type.upper()}")
    print("=" * 80)

    example = unified_df[
        unified_df["record_type"] == record_type
    ].head(2)

    display(example)


EXAMPLE: OBSERVATION


,category,collected_by,collection_date,comparable_country,confidence,evidence_basis,fiscal_year,gender,impact_direction,impact_estimate,impact_magnitude,indicator,indicator_code,indicator_direction,lag_months,location,notes,observation_date,original_text,parent_id,period_end,period_start,pillar,record_id,record_type,region,related_indicator,relationship_type,source_name,source_type,source_url,unit,value_numeric,value_text,value_type
0,NaN,2025-01-20 00:00:00,NaN,Example_Trainee,high,NaN,2014,all,NaN,NaN,NaN,Account Ownership Rate,ACC_OWNERSHIP,higher_better,NaN,national,NaN,2014-12-31,Baseline year,NaN,NaT,NaT,ACCESS,REC_0001,observation,NaN,NaN,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globalfindex,%,22.0,NaN,percentage
1,NaN,2025-01-20 00:00:00,NaN,Example_Trainee,high,NaN,2017,all,NaN,NaN,NaN,Account Ownership Rate,ACC_OWNERSHIP,higher_better,NaN,national,NaN,2017-12-31,NaN,NaN,NaT,NaT,ACCESS,REC_0002,observation,NaN,NaN,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globalfindex,%,35.0,NaN,percentage



EXAMPLE: EVENT


,category,collected_by,collection_date,comparable_country,confidence,evidence_basis,fiscal_year,gender,impact_direction,impact_estimate,impact_magnitude,indicator,indicator_code,indicator_direction,lag_months,location,notes,observation_date,original_text,parent_id,period_end,period_start,pillar,record_id,record_type,region,related_indicator,relationship_type,source_name,source_type,source_url,unit,value_numeric,value_text,value_type
33,product_launch,2025-01-20 00:00:00,NaN,Example_Trainee,high,NaN,2021,all,NaN,NaN,NaN,Telebirr Launch,EVT_TELEBIRR,NaN,NaN,national,NaN,2021-05-17,First major mobile money service in Ethiopia,NaN,NaT,NaT,NaN,EVT_0001,event,NaN,NaN,NaN,Ethio Telecom,operator,https://www.ethiotelecom.et/,NaN,NaN,Launched,categorical
34,market_entry,2025-01-20 00:00:00,NaN,Example_Trainee,high,NaN,2022,all,NaN,NaN,NaN,Safaricom Ethiopia Commercial Launch,EVT_SAFARICOM,NaN,NaN,national,NaN,2022-08-01,End of state telecom monopoly,NaN,NaT,NaT,NaN,EVT_0002,event,NaN,NaN,NaN,News,news,NaN,NaN,NaN,Launched,categorical



EXAMPLE: TARGET


,category,collected_by,collection_date,comparable_country,confidence,evidence_basis,fiscal_year,gender,impact_direction,impact_estimate,impact_magnitude,indicator,indicator_code,indicator_direction,lag_months,location,notes,observation_date,original_text,parent_id,period_end,period_start,pillar,record_id,record_type,region,related_indicator,relationship_type,source_name,source_type,source_url,unit,value_numeric,value_text,value_type
30,NaN,2025-01-20 00:00:00,NaN,Example_Trainee,high,NaN,2025,all,NaN,NaN,NaN,Account Ownership Rate,ACC_OWNERSHIP,higher_better,NaN,national,NaN,2025-12-31,NFIS-II target,NaN,NaT,NaT,ACCESS,REC_0031,target,NaN,NaN,NaN,NFIS-II Strategy,policy,https://nbe.gov.et/,%,70.0,NaN,percentage
31,NaN,2025-01-20 00:00:00,NaN,Example_Trainee,high,NaN,2028,all,NaN,NaN,NaN,Fayda Digital ID Enrollment,ACC_FAYDA,higher_better,NaN,national,NaN,2028-12-31,90M by 2028 target,NaN,NaT,NaT,ACCESS,REC_0032,target,NaN,NaN,NaN,Fayda/NIDP,policy,https://www.id.gov.et/,people,90000000.0,NaN,count



EXAMPLE: IMPACT_LINK


,category,collected_by,collection_date,comparable_country,confidence,evidence_basis,fiscal_year,gender,impact_direction,impact_estimate,impact_magnitude,indicator,indicator_code,indicator_direction,lag_months,location,notes,observation_date,original_text,parent_id,period_end,period_start,pillar,record_id,record_type,region,related_indicator,relationship_type,source_name,source_type,source_url,unit,value_numeric,value_text,value_type
43,NaN,Example_Trainee,2025-01-20 00:00:00,Kenya,medium,literature,NaN,all,increase,15.0,high,Telebirr effect on Account Ownership,NaN,NaN,12.0,national,Kenya M-Pesa showed +20pp over 5 years,2021-05-17,NaN,EVT_0001,NaN,NaN,ACCESS,IMP_0001,impact_link,NaN,ACC_OWNERSHIP,direct,NaN,NaN,NaN,%,15.0,NaN,percentage
44,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,high,empirical,NaN,all,increase,NaN,high,Telebirr effect on Telebirr Users,NaN,NaN,3.0,national,Direct subscriber acquisition,2021-05-17,NaN,EVT_0001,NaN,NaN,USAGE,IMP_0002,impact_link,NaN,USG_TELEBIRR_USERS,direct,NaN,NaN,NaN,users,NaN,NaN,count


In [10]:
pillar_summary = (
    unified_df.groupby(
        ["record_type", "pillar"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(["record_type", "record_count"], ascending=[True, False])
)

pillar_summary

,record_type,pillar,record_count
0,event,NaN,10
4,impact_link,USAGE,6
1,impact_link,ACCESS,4
2,impact_link,AFFORDABILITY,3
3,impact_link,GENDER,1
5,observation,ACCESS,14
8,observation,USAGE,11
7,observation,GENDER,4
6,observation,AFFORDABILITY,1
9,target,ACCESS,2


In [11]:
source_type_summary = (
    unified_df["source_type"]
    .fillna("Not specified")
    .value_counts()
    .rename_axis("source_type")
    .reset_index(name="record_count")
)

source_type_summary

,source_type,record_count
0,operator,15
1,Not specified,14
2,survey,10
3,regulator,7
4,research,4
5,policy,3
6,calculated,2
7,news,2


In [12]:
confidence_summary = (
    unified_df["confidence"]
    .fillna("Not specified")
    .value_counts()
    .rename_axis("confidence")
    .reset_index(name="record_count")
)

confidence_summary

,confidence,record_count
0,high,44
1,medium,13


In [13]:
observations_df = unified_df[
    unified_df["record_type"] == "observation"
].copy()

indicator_summary = (
    observations_df.groupby(
        ["pillar", "indicator_code", "indicator"],
        dropna=False
    )
    .agg(
        observation_count=("record_id", "count"),
        earliest_date=("observation_date", "min"),
        latest_date=("observation_date", "max"),
        minimum_value=("value_numeric", "min"),
        maximum_value=("value_numeric", "max")
    )
    .reset_index()
    .sort_values(["pillar", "indicator_code"])
)

indicator_summary

,pillar,indicator_code,indicator,observation_count,earliest_date,latest_date,minimum_value,maximum_value
0,ACCESS,ACC_4G_COV,4G Population Coverage,2,2023-06-30,2025-06-30,3.750000e+01,7.080000e+01
1,ACCESS,ACC_FAYDA,Fayda Digital ID Enrollment,3,2024-08-15,2025-05-15,8.000000e+06,1.500000e+07
2,ACCESS,ACC_MM_ACCOUNT,Mobile Money Account Rate,2,2021-12-31,2024-11-29,4.700000e+00,9.450000e+00
3,ACCESS,ACC_MOBILE_PEN,Mobile Subscription Penetration,1,2025-12-31,2025-12-31,6.140000e+01,6.140000e+01
4,ACCESS,ACC_OWNERSHIP,Account Ownership Rate,6,2014-12-31,2024-11-29,2.200000e+01,5.600000e+01
5,AFFORDABILITY,AFF_DATA_INCOME,Data Affordability Index,1,2024-12-31,2024-12-31,2.000000e+00,2.000000e+00
6,GENDER,GEN_GAP_ACC,Account Ownership Gender Gap,2,2021-12-31,2024-11-29,1.800000e+01,2.000000e+01
7,GENDER,GEN_GAP_MOBILE,Mobile Phone Gender Gap,1,2024-12-31,2024-12-31,2.400000e+01,2.400000e+01
8,GENDER,GEN_MM_SHARE,Female Mobile Money Account Share,1,2024-12-31,2024-12-31,1.400000e+01,1.400000e+01
9,USAGE,USG_ACTIVE_RATE,Mobile Money Activity Rate,1,2024-12-31,2024-12-31,6.600000e+01,6.600000e+01


In [17]:
date_columns = [
    "observation_date",
    "period_start",
    "period_end",
    "collection_date"
]

for column in date_columns:
    if column in unified_df.columns:
        unified_df[column] = pd.to_datetime(
            unified_df[column],
            errors="coerce"
        )

print("Date columns converted successfully.")

Date columns converted successfully.


In [16]:
observation_dates = unified_df.loc[
    unified_df["record_type"] == "observation",
    "observation_date"
].dropna()

event_dates = unified_df.loc[
    unified_df["record_type"] == "event",
    "observation_date"
].dropna()

print("Observation temporal range")
print("Earliest observation:", observation_dates.min())
print("Latest observation:", observation_dates.max())

print("\nEvent temporal range")
print("Earliest event:", event_dates.min())
print("Latest event:", event_dates.max())

Observation temporal range
Earliest observation: 2014-12-31 00:00:00
Latest observation: 2025-12-31 00:00:00

Event temporal range
Earliest event: 2021-05-17 00:00:00
Latest event: 2025-12-18 00:00:00


In [18]:
events_df = unified_df[
    unified_df["record_type"] == "event"
].copy()

event_columns = [
    "record_id",
    "observation_date",
    "category",
    "indicator",
    "source_name",
    "confidence",
    "notes"
]

events_df[event_columns].sort_values("observation_date")

,record_id,observation_date,category,indicator,source_name,confidence,notes
33,EVT_0001,2021-05-17,product_launch,Telebirr Launch,Ethio Telecom,high,NaN
41,EVT_0009,2021-09-01,policy,NFIS-II Strategy Launch,NBE,high,NaN
34,EVT_0002,2022-08-01,market_entry,Safaricom Ethiopia Commercial Launch,News,high,NaN
35,EVT_0003,2023-08-01,product_launch,M-Pesa Ethiopia Launch,Safaricom,high,NaN
36,EVT_0004,2024-01-01,infrastructure,Fayda Digital ID Program Rollout,NIDP,high,NaN
37,EVT_0005,2024-07-29,policy,Foreign Exchange Liberalization,NBE,high,NaN
38,EVT_0006,2024-10-01,milestone,P2P Transaction Count Surpasses ATM,EthSwitch,high,NaN
39,EVT_0007,2025-10-27,partnership,M-Pesa EthSwitch Integration,EthSwitch,high,NaN
42,EVT_0010,2025-12-15,pricing,Safaricom Ethiopia Price Increase,News,high,NaN
40,EVT_0008,2025-12-18,infrastructure,EthioPay Instant Payment System Launch,NBE/EthSwitch,high,NaN


In [19]:
impact_links_df = unified_df[
    unified_df["record_type"] == "impact_link"
].copy()

impact_columns = [
    "record_id",
    "parent_id",
    "pillar",
    "related_indicator",
    "impact_direction",
    "impact_magnitude",
    "lag_months",
    "evidence_basis",
    "confidence"
]

impact_links_df[impact_columns]

,record_id,parent_id,pillar,related_indicator,impact_direction,impact_magnitude,lag_months,evidence_basis,confidence
43,IMP_0001,EVT_0001,ACCESS,ACC_OWNERSHIP,increase,high,12.0,literature,medium
44,IMP_0002,EVT_0001,USAGE,USG_TELEBIRR_USERS,increase,high,3.0,empirical,high
45,IMP_0003,EVT_0001,USAGE,USG_P2P_COUNT,increase,high,6.0,empirical,medium
46,IMP_0004,EVT_0002,ACCESS,ACC_4G_COV,increase,medium,12.0,empirical,medium
47,IMP_0005,EVT_0002,AFFORDABILITY,AFF_DATA_INCOME,decrease,medium,12.0,literature,medium
48,IMP_0006,EVT_0003,USAGE,USG_MPESA_USERS,increase,high,3.0,empirical,high
49,IMP_0007,EVT_0003,ACCESS,ACC_MM_ACCOUNT,increase,medium,6.0,theoretical,medium
50,IMP_0008,EVT_0004,ACCESS,ACC_OWNERSHIP,increase,medium,24.0,literature,medium
51,IMP_0009,EVT_0004,GENDER,GEN_GAP_ACC,decrease,medium,24.0,literature,medium
52,IMP_0010,EVT_0005,AFFORDABILITY,AFF_DATA_INCOME,increase,high,3.0,empirical,high


In [22]:
event_details = events_df[
    [
        "record_id",
        "indicator",
        "observation_date",
        "category"
    ]
].rename(
    columns={
        "record_id": "event_record_id",
        "indicator": "event_name",
        "observation_date": "event_date",
        "category": "event_category"
    }
)

impact_event_summary = impact_links_df.merge(
    event_details,
    left_on="parent_id",
    right_on="event_record_id",
    how="left"
)

impact_event_summary[
    [
        "record_id",
        "parent_id",
        "event_name",
        "event_date",
        "event_category",
        "pillar",
        "related_indicator",
        "impact_direction",
        "impact_magnitude",
        "lag_months"
    ]
].sort_values(
    ["event_date", "pillar"]
)

,record_id,parent_id,event_name,event_date,event_category,pillar,related_indicator,impact_direction,impact_magnitude,lag_months
0,IMP_0001,EVT_0001,Telebirr Launch,2021-05-17,product_launch,ACCESS,ACC_OWNERSHIP,increase,high,12.0
1,IMP_0002,EVT_0001,Telebirr Launch,2021-05-17,product_launch,USAGE,USG_TELEBIRR_USERS,increase,high,3.0
2,IMP_0003,EVT_0001,Telebirr Launch,2021-05-17,product_launch,USAGE,USG_P2P_COUNT,increase,high,6.0
3,IMP_0004,EVT_0002,Safaricom Ethiopia Commercial Launch,2022-08-01,market_entry,ACCESS,ACC_4G_COV,increase,medium,12.0
4,IMP_0005,EVT_0002,Safaricom Ethiopia Commercial Launch,2022-08-01,market_entry,AFFORDABILITY,AFF_DATA_INCOME,decrease,medium,12.0
6,IMP_0007,EVT_0003,M-Pesa Ethiopia Launch,2023-08-01,product_launch,ACCESS,ACC_MM_ACCOUNT,increase,medium,6.0
5,IMP_0006,EVT_0003,M-Pesa Ethiopia Launch,2023-08-01,product_launch,USAGE,USG_MPESA_USERS,increase,high,3.0
7,IMP_0008,EVT_0004,Fayda Digital ID Program Rollout,2024-01-01,infrastructure,ACCESS,ACC_OWNERSHIP,increase,medium,24.0
8,IMP_0009,EVT_0004,Fayda Digital ID Program Rollout,2024-01-01,infrastructure,GENDER,GEN_GAP_ACC,decrease,medium,24.0
9,IMP_0010,EVT_0005,Foreign Exchange Liberalization,2024-07-29,policy,AFFORDABILITY,AFF_DATA_INCOME,increase,high,3.0


In [23]:
unmatched_impacts = impact_event_summary[
    impact_event_summary["event_name"].isna()
]

print("Impact links without a matching event:", len(unmatched_impacts))

if not unmatched_impacts.empty:
    display(
        unmatched_impacts[
            ["record_id", "parent_id", "related_indicator"]
        ]
    )
else:
    print("All impact links are successfully connected to an event.")

Impact links without a matching event: 0
All impact links are successfully connected to an event.


## 3. Data-Quality Assessment

The following checks identify missing documentation, incomplete source information, invalid categorical values, duplicate identifiers, and records that do not follow the unified schema.

In [24]:
missing_summary = (
    unified_df.isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(unified_df) * 100
).round(2)

missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

missing_summary.head(20)

,column,missing_count,missing_percentage
25,region,57,100.00
33,value_text,47,82.46
0,category,47,82.46
20,period_end,47,82.46
21,period_start,47,82.46
9,impact_estimate,45,78.95
2,collection_date,43,75.44
16,notes,43,75.44
5,evidence_basis,43,75.44
8,impact_direction,43,75.44


In [25]:
documentation_fields = [
    "source_name",
    "source_url",
    "original_text",
    "confidence",
    "collected_by",
    "collection_date",
    "notes"
]

documentation_quality = []

for column in documentation_fields:
    documentation_quality.append({
        "field": column,
        "completed_records": unified_df[column].notna().sum(),
        "missing_records": unified_df[column].isna().sum(),
        "completion_rate_percent": round(
            unified_df[column].notna().mean() * 100,
            2
        )
    })

documentation_quality_df = pd.DataFrame(documentation_quality)

documentation_quality_df

,field,completed_records,missing_records,completion_rate_percent
0,source_name,43,14,75.44
1,source_url,31,26,54.39
2,original_text,33,24,57.89
3,confidence,57,0,100.00
4,collected_by,57,0,100.00
5,collection_date,14,43,24.56
6,notes,14,43,24.56


In [26]:
collector_counts = (
    unified_df["collected_by"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("collected_by")
    .reset_index(name="record_count")
)

collector_counts

,collected_by,record_count
0,2025-01-20 00:00:00,43
1,Example_Trainee,14


In [27]:
duplicate_ids = unified_df[
    unified_df["record_id"].duplicated(keep=False)
].sort_values("record_id")

print("Number of duplicated record IDs:", len(duplicate_ids))

if not duplicate_ids.empty:
    display(duplicate_ids)
else:
    print("No duplicated record IDs were found.")

Number of duplicated record IDs: 0
No duplicated record IDs were found.


In [28]:
events_with_pillars = unified_df[
    (unified_df["record_type"] == "event") &
    (unified_df["pillar"].notna())
]

print(
    "Events incorrectly assigned directly to a pillar:",
    len(events_with_pillars)
)

if not events_with_pillars.empty:
    display(
        events_with_pillars[
            ["record_id", "indicator", "category", "pillar"]
        ]
    )
else:
    print("All events follow the neutral-event design rule.")

Events incorrectly assigned directly to a pillar: 0
All events follow the neutral-event design rule.


In [30]:
required_fields = {
    "observation": [
        "record_id",
        "pillar",
        "indicator",
        "indicator_code",
        "value_numeric",
        "observation_date",
        "source_name",
        "confidence"
    ],
    "event": [
        "record_id",
        "category",
        "indicator",
        "observation_date",
        "source_name",
        "confidence"
    ],
    "target": [
        "record_id",
        "pillar",
        "indicator",
        "indicator_code",
        "value_numeric",
        "source_name"
    ],
    "impact_link": [
        "record_id",
        "parent_id",
        "pillar",
        "related_indicator",
        "impact_direction",
        "impact_magnitude",
        "lag_months",
        "evidence_basis"
    ]
}

quality_results = []

for record_type, fields in required_fields.items():
    subset = unified_df[
        unified_df["record_type"] == record_type
    ]

    for field in fields:
        quality_results.append({
            "record_type": record_type,
            "required_field": field,
            "record_count": len(subset),
            "missing_count": subset[field].isna().sum(),
            "completion_rate_percent": round(
                subset[field].notna().mean() * 100,
                2
            )
        })

required_field_quality_df = pd.DataFrame(quality_results)

required_field_quality_df[
    required_field_quality_df["missing_count"] > 0
].sort_values(
    ["record_type", "missing_count"],
    ascending=[True, False]
)

,record_type,required_field,record_count,missing_count,completion_rate_percent


In [31]:
print(unified_df.columns.tolist())

['category', 'collected_by', 'collection_date', 'comparable_country', 'confidence', 'evidence_basis', 'fiscal_year', 'gender', 'impact_direction', 'impact_estimate', 'impact_magnitude', 'indicator', 'indicator_code', 'indicator_direction', 'lag_months', 'location', 'notes', 'observation_date', 'original_text', 'parent_id', 'period_end', 'period_start', 'pillar', 'record_id', 'record_type', 'region', 'related_indicator', 'relationship_type', 'source_name', 'source_type', 'source_url', 'unit', 'value_numeric', 'value_text', 'value_type']


In [32]:
indicator_coverage = (
    observations_df.groupby(
        ["pillar", "indicator_code", "indicator"],
        dropna=False
    )
    .agg(
        observation_count=("record_id", "count"),
        first_observation=("observation_date", "min"),
        last_observation=("observation_date", "max")
    )
    .reset_index()
)

indicator_coverage["coverage_status"] = np.select(
    [
        indicator_coverage["observation_count"] == 1,
        indicator_coverage["observation_count"].between(2, 3),
        indicator_coverage["observation_count"] >= 4
    ],
    [
        "Single observation",
        "Sparse",
        "Relatively stronger"
    ],
    default="Unknown"
)

indicator_coverage.sort_values(
    ["observation_count", "pillar"]
)

,pillar,indicator_code,indicator,observation_count,first_observation,last_observation,coverage_status
3,ACCESS,ACC_MOBILE_PEN,Mobile Subscription Penetration,1,2025-12-31,2025-12-31,Single observation
5,AFFORDABILITY,AFF_DATA_INCOME,Data Affordability Index,1,2024-12-31,2024-12-31,Single observation
7,GENDER,GEN_GAP_MOBILE,Mobile Phone Gender Gap,1,2024-12-31,2024-12-31,Single observation
8,GENDER,GEN_MM_SHARE,Female Mobile Money Account Share,1,2024-12-31,2024-12-31,Single observation
9,USAGE,USG_ACTIVE_RATE,Mobile Money Activity Rate,1,2024-12-31,2024-12-31,Single observation
10,USAGE,USG_ATM_COUNT,ATM Transaction Count,1,2025-07-07,2025-07-07,Single observation
11,USAGE,USG_ATM_VALUE,ATM Transaction Value,1,2025-07-07,2025-07-07,Single observation
12,USAGE,USG_CROSSOVER,P2P/ATM Crossover Ratio,1,2025-07-07,2025-07-07,Single observation
13,USAGE,USG_MPESA_ACTIVE,M-Pesa 90-Day Active Users,1,2024-12-31,2024-12-31,Single observation
14,USAGE,USG_MPESA_USERS,M-Pesa Registered Users,1,2024-12-31,2024-12-31,Single observation


In [33]:
account_ownership = observations_df[
    observations_df["indicator_code"] == "ACC_OWNERSHIP"
].copy()

account_ownership[
    [
        "record_id",
        "observation_date",
        "value_numeric",
        "source_name",
        "confidence"
    ]
].sort_values("observation_date")

,record_id,observation_date,value_numeric,source_name,confidence
0,REC_0001,2014-12-31,22.0,Global Findex 2014,high
1,REC_0002,2017-12-31,35.0,Global Findex 2017,high
2,REC_0003,2021-12-31,46.0,Global Findex 2021,high
3,REC_0004,2021-12-31,56.0,Global Findex 2021,high
4,REC_0005,2021-12-31,36.0,Global Findex 2021,high
5,REC_0006,2024-11-29,49.0,Global Findex 2024,high


### Initial Data-Quality Findings

The initial review identified the following issues:

1. The workbook contains 57 records: 30 observations, 10 events, 14 impact links, and 3 targets.
2. Event records correctly leave the `pillar` field empty and use impact links to represent affected dimensions.
3. Several documentation fields, including source URLs and original source text, are incomplete.
4. Some records still use the placeholder collector name `Example_Trainee`.
5. Many indicators contain only one or two observations, limiting conventional time-series modeling.
6. The Account Ownership series does not include the 2011 Global Findex baseline of 14%, although the assignment requires analysis from 2011 to 2024.
7. Operator-reported registered accounts and survey-reported adult usage measure different concepts and must not be interpreted as equivalent.

## 4. Dataset Cleaning and Enrichment

The starter dataset contains a documentation alignment issue in the `collected_by` and `collection_date` fields. Some records contain a date in the collector field, while impact-link records use the placeholder name `Example_Trainee`.

The cleaning process will:

1. Move date-like collector values into `collection_date`.
2. Replace placeholder collector values with the analyst's name.
3. Preserve the original raw dataset.
4. Add the missing 2011 Account Ownership observation.
5. Validate the enriched dataset before export.

In [34]:
cleaned_df = unified_df.copy()

print("Records before cleaning:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))

Records before cleaning: 57
Columns: 35


In [35]:
# Convert any date-like values found in collected_by.
collector_as_date = pd.to_datetime(
    cleaned_df["collected_by"],
    errors="coerce"
)

date_in_collector_mask = collector_as_date.notna()

print(
    "Records containing a date in collected_by:",
    date_in_collector_mask.sum()
)

# Move those dates into collection_date when collection_date is missing.
cleaned_df.loc[
    date_in_collector_mask & cleaned_df["collection_date"].isna(),
    "collection_date"
] = collector_as_date[
    date_in_collector_mask & cleaned_df["collection_date"].isna()
]

# Replace all incorrect or placeholder collector values.
incorrect_collector_mask = (
    date_in_collector_mask |
    cleaned_df["collected_by"].isna() |
    cleaned_df["collected_by"].astype(str).str.contains(
        "Example_Trainee",
        case=False,
        na=False
    )
)

cleaned_df.loc[
    incorrect_collector_mask,
    "collected_by"
] = "Bilen M. Gebremariam"

# Standardize the collection-date column.
cleaned_df["collection_date"] = pd.to_datetime(
    cleaned_df["collection_date"],
    errors="coerce"
)

print("\nCollector values after correction:")
display(
    cleaned_df["collected_by"]
    .value_counts(dropna=False)
    .rename_axis("collected_by")
    .reset_index(name="record_count")
)

print(
    "\nMissing collection dates:",
    cleaned_df["collection_date"].isna().sum()
)

Records containing a date in collected_by: 43

Collector values after correction:


,collected_by,record_count
0,Bilen M. Gebremariam,57



Missing collection dates: 0


In [36]:
existing_access_records = cleaned_df[
    (cleaned_df["record_type"] == "observation") &
    (cleaned_df["indicator_code"] == "ACC_OWNERSHIP")
][
    [
        "record_id",
        "pillar",
        "indicator",
        "indicator_code",
        "value_numeric",
        "observation_date",
        "gender",
        "source_name"
    ]
].sort_values(
    ["observation_date", "gender"],
    na_position="first"
)

existing_access_records

,record_id,pillar,indicator,indicator_code,value_numeric,observation_date,gender,source_name
0,REC_0001,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,22.0,2014-12-31,all,Global Findex 2014
1,REC_0002,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,35.0,2017-12-31,all,Global Findex 2017
2,REC_0003,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,46.0,2021-12-31,all,Global Findex 2021
4,REC_0005,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,36.0,2021-12-31,female,Global Findex 2021
3,REC_0004,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,56.0,2021-12-31,male,Global Findex 2021
5,REC_0006,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,49.0,2024-11-29,all,Global Findex 2024


In [42]:
new_record_id = "REC_0058"

if new_record_id in cleaned_df["record_id"].values:
    raise ValueError(
        f"{new_record_id} already exists. Choose another record ID."
    )

print(f"{new_record_id} is available.")

REC_0058 is available.


In [39]:
cleaned_df[
    cleaned_df["record_id"] == "REC_0031"
]


,category,collected_by,collection_date,comparable_country,confidence,evidence_basis,fiscal_year,gender,impact_direction,impact_estimate,impact_magnitude,indicator,indicator_code,indicator_direction,lag_months,location,notes,observation_date,original_text,parent_id,period_end,period_start,pillar,record_id,record_type,region,related_indicator,relationship_type,source_name,source_type,source_url,unit,value_numeric,value_text,value_type
30,NaN,Bilen M. Gebremariam,2025-01-20,Example_Trainee,high,NaN,2025,all,NaN,NaN,NaN,Account Ownership Rate,ACC_OWNERSHIP,higher_better,NaN,national,NaN,2025-12-31,NFIS-II target,NaN,NaT,NaT,ACCESS,REC_0031,target,NaN,NaN,NaN,NFIS-II Strategy,policy,https://nbe.gov.et/,%,70.0,NaN,percentage


In [40]:
cleaned_df[
    cleaned_df["record_id"] == "REC_0031"
][[
    "record_id",
    "record_type",
    "indicator_code",
    "value_numeric",
    "observation_date"
]]

,record_id,record_type,indicator_code,value_numeric,observation_date
30,REC_0031,target,ACC_OWNERSHIP,70.0,2025-12-31


In [41]:
new_record_id = "REC_0058"

In [43]:
collection_date = pd.Timestamp.today().normalize()

new_2011_observation = {
    "record_id": "REC_0058",
    "record_type": "observation",
    "pillar": "ACCESS",
    "indicator": "Account Ownership Rate",
    "indicator_code": "ACC_OWNERSHIP",
    "value_numeric": 14.0,
    "value_text": None,
    "value_type": "percentage",
    "unit": "percent",
    "observation_date": pd.Timestamp("2011-12-31"),
    "period_start": pd.Timestamp("2011-01-01"),
    "period_end": pd.Timestamp("2011-12-31"),
    "fiscal_year": 2011,
    "gender": None,
    "location": "Ethiopia",
    "region": None,
    "category": None,
    "parent_id": None,
    "relationship_type": None,
    "related_indicator": None,
    "impact_direction": None,
    "impact_magnitude": None,
    "impact_estimate": None,
    "lag_months": None,
    "evidence_basis": None,
    "indicator_direction": "increase",
    "source_name": "World Bank Global Findex 2011",
    "source_type": "survey",
    "source_url": (
        "https://datatopics.worldbank.org/"
        "financialinclusion/country/ethiopia/"
    ),
    "original_text": (
        "Ethiopia's account ownership rate in 2011 was 14% "
        "of adults aged 15 and above."
    ),
    "confidence": "high",
    "collected_by": "Bilen M. Gebremariam",
    "collection_date": collection_date,
    "notes": (
        "Added as the missing baseline required to analyze Ethiopia's "
        "account ownership trajectory from 2011 to 2024."
    ),
    "comparable_country": None
}

new_2011_df = pd.DataFrame(
    [new_2011_observation]
).reindex(columns=cleaned_df.columns)

cleaned_df = pd.concat(
    [cleaned_df, new_2011_df],
    ignore_index=True
)

print("Records after adding 2011 observation:", len(cleaned_df))

Records after adding 2011 observation: 58


In [45]:
account_ownership_trajectory = cleaned_df[
    (cleaned_df["record_type"] == "observation") &
    (cleaned_df["indicator_code"] == "ACC_OWNERSHIP") &
    (cleaned_df["gender"].isna())
][
    [
        "record_id",
        "observation_date",
        "value_numeric",
        "unit",
        "source_name",
        "confidence"
    ]
].sort_values("observation_date")

account_ownership_trajectory

,record_id,observation_date,value_numeric,unit,source_name,confidence
57,REC_0058,2011-12-31,14.0,percent,World Bank Global Findex 2011,high


In [46]:
validation_results = {
    "total_records": len(cleaned_df),
    "duplicate_record_ids": cleaned_df[
        "record_id"
    ].duplicated().sum(),
    "missing_record_ids": cleaned_df[
        "record_id"
    ].isna().sum(),
    "missing_record_types": cleaned_df[
        "record_type"
    ].isna().sum(),
    "missing_collector_names": cleaned_df[
        "collected_by"
    ].isna().sum(),
    "missing_collection_dates": cleaned_df[
        "collection_date"
    ].isna().sum(),
    "placeholder_collectors": cleaned_df[
        "collected_by"
    ].astype(str).str.contains(
        "Example_Trainee",
        case=False,
        na=False
    ).sum(),
    "account_ownership_points": len(
        cleaned_df[
            (cleaned_df["record_type"] == "observation") &
            (cleaned_df["indicator_code"] == "ACC_OWNERSHIP") &
            (cleaned_df["gender"].isna())
        ]
    )
}

validation_results_df = pd.DataFrame(
    validation_results.items(),
    columns=["validation_check", "result"]
)

validation_results_df

,validation_check,result
0,total_records,58
1,duplicate_record_ids,0
2,missing_record_ids,0
3,missing_record_types,0
4,missing_collector_names,0
5,missing_collection_dates,0
6,placeholder_collectors,0
7,account_ownership_points,1


In [47]:
OUTPUT_CSV_PATH = (
    PROCESSED_DATA_DIR /
    "ethiopia_fi_enriched.csv"
)

OUTPUT_EXCEL_PATH = (
    PROCESSED_DATA_DIR /
    "ethiopia_fi_enriched.xlsx"
)

cleaned_df.to_csv(
    OUTPUT_CSV_PATH,
    index=False
)

cleaned_df.to_excel(
    OUTPUT_EXCEL_PATH,
    index=False
)

print("CSV exported to:")
print(OUTPUT_CSV_PATH)

print("\nExcel file exported to:")
print(OUTPUT_EXCEL_PATH)

print("\nExported records:", len(cleaned_df))


CSV exported to:
c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\data\processed\ethiopia_fi_enriched.csv

Excel file exported to:
c:\Users\gebre\OneDrive\Documents\Programming\KIAM 10 academy\Week 11\ethiopia-fi-forecast\data\processed\ethiopia_fi_enriched.xlsx

Exported records: 58
